The goal of this notebook is to derive a "signal" metric for the spatial predictions. 

For that matter, the number of high confidence cells per cell type in the spatial dataset that went into the smoothing as the number of high confidence cells will be counted and collected.

High confidence cells are defined by label propagation confidences > 0.5. 

In [1]:
from pathlib import Path 

import pandas as pd
import scanpy as sc
import numpy as np
from scipy import sparse

from dotenv import dotenv_values

In [2]:
# Setting up filepaths from config
CONFIG = dotenv_values("config.env") 
DATADIR = Path(CONFIG['DATADIR'])

SPATIALDIR = DATADIR / "7_spatial"

ADATA_FP = SPATIALDIR / "SP_concat.h5ad"

OUTDIR = SPATIALDIR / "analyses"
OUTDIR.mkdir(exist_ok=True, parents=True)

In [3]:
HI_CONF_THR = 0.5 # high confidence threshold

In [4]:
adata = sc.read_h5ad(ADATA_FP, backed="r")

Pathmat are the paths through our clustering tree from L1 to L5, while adata.obsm["P_path"] is shape (n_cells, n leaf (ie L5) clusters).

Hence, in the loop below, path probabilities must be summed up for all parent clusters of L5 clusters when assessing L4 clusters for instance. 

Afterwards, it is checked whether the resulting probabilities exceed the above high confidence threshold.

In [5]:
pathmat = pd.DataFrame(data=adata.uns["pathmat"], columns=[f"L{i}" for i in range(1,6)])

In [6]:
pathmat

,L1,L2,L3,L4,L5
0,Gb1-2 GABA,Gb2-39 SI-MPO-LPO Lhx8,Gb3-124 MPO Meis1,Gb4-319 MPO Meis1.Rskr,Gb5-428 MPO Meis1.Rskr.Cdh20
1,Gb1-2 GABA,Gb2-39 SI-MPO-LPO Lhx8,Gb3-124 MPO Meis1,Gb4-320 MPO Meis1.Popdc3,Gb5-429 MPO Meis1.Popdc3.Epha7
2,Gb1-2 GABA,Gb2-39 SI-MPO-LPO Lhx8,Gb3-124 MPO Meis1,Gb4-321 MPO Meis1.Esr1,Gb5-430 MPO Meis1.Mc4r
3,Gb1-2 GABA,Gb2-39 SI-MPO-LPO Lhx8,Gb3-124 MPO Meis1,Gb4-321 MPO Meis1.Esr1,Gb5-431 MPO Meis1.Esr1
4,Gb1-2 GABA,Gb2-39 SI-MPO-LPO Lhx8,Gb3-124 MPO Meis1,Gb4-322 MPO Meis1.Myo5b,Gb5-432 MPO Meis1.Myo5b.Ntng1
...,...,...,...,...,...
927,NN1-4 Astro-Epen,NN2-13 Tanycyte,NN3-27 Tanycyte_alpha1,NN4-42 Tanycyte_alpha1-2,NN5-42 Tanycyte_alpha1-2
928,NN1-4 Astro-Epen,NN2-13 Tanycyte,NN3-27 Tanycyte_alpha1,NN4-43 Tanycyte_alpha1-1,NN5-43 Tanycyte_alpha1-1
929,NN1-4 Astro-Epen,NN2-14 Ependymal,NN3-28 Ependymal_1,NN4-44 Ependymal_1-2,NN5-44 Ependymal_1-2
930,NN1-4 Astro-Epen,NN2-14 Ependymal,NN3-28 Ependymal_1,NN4-45 Ependymal_3-1,NN5-45 Ependymal_3-1


In [7]:

# Ensure P_path is CSC format for efficient column slicing
P_path = adata.obsm["P_path"]
if sparse.issparse(P_path):
    P_path = P_path.tocsc()  # CSC is efficient for column indexing
else:
    P_path = sparse.csc_matrix(P_path)

rows = []

for level in ["L1", "L2", "L3", "L4", "L5"]:
    clusters = pathmat[level].unique()
    
    for cluster in clusters:
        # Find which path indices (columns) pass through this cluster
        path_indices = pathmat.index[pathmat[level] == cluster].tolist()
        
        # Sum probabilities across relevant columns — stays sparse
        cluster_probs = P_path[:, path_indices].sum(axis=1)  # shape: (n_cells, 1), np.matrix
        
        # Count cells exceeding threshold — np.matrix.A1 flattens to 1D array
        n_hi_conf = (cluster_probs.A1 > HI_CONF_THR).sum()
        
        rows.append({"level": level, "cluster": cluster, f"n_cells_presmooth_p_gt_{HI_CONF_THR}": n_hi_conf})

results_df = (
    pd.DataFrame(rows)
    .sort_values(["level", "cluster"])
    .reset_index(drop=True)
)

In [8]:
results_df

,level,cluster,n_cells_presmooth_p_gt_0.5
0,L1,Gb1-2 GABA,433607
1,L1,Gl1-1 Glut,256902
2,L1,NN1-1 Oligo,231723
3,L1,NN1-2 Immune,21381
4,L1,NN1-3 Vascular,129944
...,...,...,...
2005,L5,NN5-5 Oligo 2_1-2,1131
2006,L5,NN5-6 OPC_1-1,24509
2007,L5,NN5-7 OPC_2-1,3497
2008,L5,NN5-8 Microglia_1-1,970


In [9]:
results_df.to_csv(OUTDIR / "high_confidence_cells.csv", index=False)